# insurance-deploy

**Champion/challenger pricing framework for UK insurance — model registry, quote routing, ENBP audit logging, and statistical promotion tests.**

This notebook runs the full deployment lifecycle on synthetic motor data: register two pricing models, set up a shadow-mode experiment, simulate a quote campaign with ENBP logging, compute KPIs, and generate an ICOBS 6B.2.51R audit report.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-deploy/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-deploy numpy

## 1. Synthetic pricing models

We need two objects with a `.predict()` method — one champion and one challenger. In production these would be CatBoost or GLM objects loaded from your model artefact store. Here we use minimal stubs so the notebook is self-contained and runs in under a minute.

In [ ]:
import numpy as np
from datetime import date

rng = np.random.default_rng(42)

class _SyntheticPricingModel:
    """Minimal stub — returns a log-normal quote given a dict of risk factors."""
    def __init__(self, base_premium: float, seed: int):
        self._base = base_premium
        self._rng = np.random.default_rng(seed)

    def predict(self, X):
        n = len(X) if hasattr(X, '__len__') else 1
        return self._rng.lognormal(np.log(self._base), 0.15, n)

champion_model = _SyntheticPricingModel(base_premium=420.0, seed=1)
challenger_model = _SyntheticPricingModel(base_premium=415.0, seed=2)  # slightly cheaper on average

print("Champion sample quotes:", champion_model.predict([[]] * 3).round(2))
print("Challenger sample quotes:", challenger_model.predict([[]] * 3).round(2))

## 2. Register both models

`ModelRegistry` stores each model as a joblib file with a SHA-256 hash for tamper detection. The registry is append-only — you cannot overwrite a registered version. This is the audit trail requirement: you must be able to reconstruct which exact model object priced each quote.

In [ ]:
import tempfile
import os
from insurance_deploy import ModelRegistry

# Use a temp directory so the notebook is self-contained
tmpdir = tempfile.mkdtemp()
registry = ModelRegistry(os.path.join(tmpdir, "registry"))

champion_mv = registry.register(
    champion_model,
    name="motor",
    version="2.0",
    metadata={
        "training_date": "2024-01-01",
        "features": ["age", "ncd", "postcode_band"],
        "holdout_gini": 0.42,
        "description": "GLM Poisson frequency + Gamma severity",
    },
)

challenger_mv = registry.register(
    challenger_model,
    name="motor",
    version="3.0",
    metadata={
        "training_date": "2024-07-01",
        "features": ["age", "ncd", "postcode_band", "vehicle_value"],
        "holdout_gini": 0.45,
        "description": "CatBoost Tweedie with vehicle value factor",
    },
)

print("Registered versions:")
for mv in registry.list("motor"):
    print(f"  {mv}")

## 3. Set up the shadow-mode experiment

Shadow mode is the default and the right choice for most teams. The challenger scores every quote in parallel, but the customer always sees the champion price. Zero FCA Consumer Duty regulatory risk, and no adverse selection contaminating your eventual loss ratio comparison.

Routing is deterministic: `SHA-256(policy_id + experiment_name) mod 100`. The same policy always routes to the same arm. No database of assignments needed — any routing decision can be verified from first principles.

In [ ]:
from insurance_deploy import Experiment

exp = Experiment(
    name="motor_v3_vs_v2",
    champion=champion_mv,
    challenger=challenger_mv,
    challenger_pct=0.20,  # 20% routed to challenger bucket (shadow mode: does not affect pricing)
    mode="shadow",
)

print(exp)

# Demonstrate routing determinism
policy = "POL-00042"
arm1 = exp.route(policy)
arm2 = exp.route(policy)
print(f"\nRouting for {policy}: {arm1} (first call), {arm2} (second call) -- always the same")

## 4. Simulate a quote campaign with ENBP logging

We simulate 500 motor renewal quotes. For each quote we:
1. Route the policy to champion or challenger (for record-keeping)
2. Price using champion (shadow mode -- champion always prices)
3. Log the quote with ENBP for renewals (ICOBS 6B.2.51R requirement)
4. Simulate bind decisions (30% hit rate)

In production, ENBP is calculated by your pricing team. The library records the value; it does not derive it.

In [ ]:
from insurance_deploy import QuoteLogger

logger = QuoteLogger(os.path.join(tmpdir, "quotes.db"))

n_quotes = 500
policy_ids = [f"POL-{i:05d}" for i in range(n_quotes)]
is_renewal = rng.random(n_quotes) < 0.6  # 60% renewals

bound_policies = []

for i, policy_id in enumerate(policy_ids):
    arm = exp.route(policy_id)

    # Champion always prices in shadow mode
    champion_price = float(champion_model.predict([[]])[0])
    # Challenger scores but price is not shown
    challenger_price = float(challenger_model.predict([[]])[0])

    # ENBP: last year's renewal price, capped at new business price
    # In practice your pricing system computes this. Here we simulate it.
    enbp = champion_price * rng.uniform(0.98, 1.05) if is_renewal[i] else None

    logger.log_quote(
        policy_id=policy_id,
        experiment_name=exp.name,
        arm=arm,
        model_version=champion_mv.version_id,  # champion prices in shadow
        quoted_price=champion_price,
        enbp=enbp,
        renewal_flag=bool(is_renewal[i]),
    )

    # Simulate bind: higher-priced quotes convert less
    if rng.random() < 0.30:
        logger.log_bind(policy_id, bound_price=champion_price)
        bound_policies.append(policy_id)

print(f"Logged {logger.quote_count(exp.name)} quotes")
print(f"Bound policies: {len(bound_policies)}")
print(f"Overall hit rate: {len(bound_policies) / n_quotes:.1%}")

## 5. KPI tracking

KPIs split by arm. In shadow mode, any difference in hit rate between arms reflects cohort assignment, not different pricing. Hit rate differences only mean anything in live mode where the challenger prices its fraction.

In [ ]:
from insurance_deploy import KPITracker

tracker = KPITracker(logger)

hr = tracker.hit_rate(exp.name)
print("Hit rate by arm:")
for arm, stats in hr.items():
    print(f"  {arm}: {stats['quoted']} quoted, {stats['bound']} bound, "
          f"{stats['hit_rate']:.1%} hit rate")

vol = tracker.quote_volume(exp.name)
print("\nQuote volume by arm:")
for arm, stats in vol.items():
    print(f"  {arm}: n={stats['n']}, mean price=£{stats['mean_price']:.0f}")

## 6. Power analysis

Before committing to an experiment, run the power analysis. It tells you how many months until you have enough data to make a statistically credible promotion decision. The answer is usually longer than stakeholders expect.

In [ ]:
pa = tracker.power_analysis(
    exp.name,
    target_delta_lr=0.03,   # detecting a 3pp loss ratio improvement
    monthly_quote_volume=n_quotes,  # quotes per month in live
)
print("Power analysis (3pp LR improvement target):")
for k, v in pa.items():
    print(f"  {k}: {v}")

## 7. ENBP audit report

The ICOBS 6B.2.51R audit report. Designed for inclusion in the SMF holder's annual attestation pack. The report shows ENBP compliance by arm, breach count, and the renewal quote breakdown.

In [ ]:
from insurance_deploy import ENBPAuditReport

reporter = ENBPAuditReport(logger)
md = reporter.generate(
    experiment_name=exp.name,
    period_start="2024-01-01",
    period_end="2024-12-31",
    firm_name="Acme Motor Insurance Ltd",
    smf_holder="Jane Smith",
)
print(md)

## 8. Inspect the audit log as a Polars DataFrame

The full audit log is available as Polars or pandas. One row per quote, permanently recorded.

In [ ]:
import polars as pl

quotes_df = logger.to_polars("quotes")
print(f"Quotes table: {quotes_df.shape[0]} rows, {quotes_df.shape[1]} columns")
print(quotes_df.select(["policy_id", "arm", "model_version", "quoted_price", "enbp", "enbp_flag", "renewal_flag"]).head(8))

## What you should see

- Quote log with 500 rows, one per policy, with ENBP flags for renewals
- Hit rate split ~30% across both arms (no meaningful difference expected in shadow mode)
- Power analysis showing 15-25+ months to loss ratio significance including 12-month development tail
- ENBP audit report in Markdown, ready to paste into an attestation pack

The long timeline from the power analysis is not a limitation of the library. Developed loss ratio has a 12-month reward tail. Any framework claiming to optimise on LR signal faster than this is using a proxy metric or incorrect assumptions.

## Next steps

- **`ModelComparison`** -- bootstrap likelihood-ratio test and hit rate z-test once you have 12+ months of data
- **`mode='live'`** -- routes the challenger fraction to see the challenger price (requires legal sign-off for Consumer Duty)
- **Radar wrapper pattern** -- see README for integrating as a governance layer around WTW Radar Live

**GitHub:** https://github.com/burning-cost/insurance-deploy  
**PyPI:** https://pypi.org/project/insurance-deploy/